In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes where present
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- 1) metric.csv summaries ---
# cmdb_id counts
metric_cmdb_counts = metric_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# kpi_name counts (top 20)
metric_kpi_counts_top20 = metric_df.groupby('kpi_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

# ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# global P95 per (cmdb_id, kpi_name)
metric_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
metric_p95_top20 = metric_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

# Keep outputs compact: select specified columns
metric_cmdb_counts = metric_cmdb_counts[['cmdb_id','count']]
metric_kpi_counts_top20 = metric_kpi_counts_top20[['kpi_name','count']]
metric_p95_top20 = metric_p95_top20[['cmdb_id','kpi_name','p95']]

# --- 2) trace.csv summaries ---
# cmdb_id counts
trace_cmdb_counts = trace_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# trace_name counts (top 20)
trace_name_counts_top20 = trace_df.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

# numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# global P95 per (cmdb_id, trace_name)
trace_p95 = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index(name='p95')
trace_p95_top20 = trace_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

trace_cmdb_counts = trace_cmdb_counts[['cmdb_id','count']]
trace_name_counts_top20 = trace_name_counts_top20[['trace_name','count']]
trace_p95_top20 = trace_p95_top20[['cmdb_id','trace_name','p95']]

# --- 3) log.csv summaries ---
# cmdb_id counts
log_cmdb_counts = log_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# log_name counts (top 20)
log_name_counts_top20 = log_df.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

# numeric values
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# global P95 per (cmdb_id, log_name)
log_p95 = log_df.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.95).reset_index(name='p95')
log_p95_top20 = log_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

log_cmdb_counts = log_cmdb_counts[['cmdb_id','count']]
log_name_counts_top20 = log_name_counts_top20[['log_name','count']]
log_p95_top20 = log_p95_top20[['cmdb_id','log_name','p95']]

# --- 4) error_logs.csv summaries ---
# counts per cmdb_id and earliest/latest timestamps
if not error_df.empty:
    error_summary = error_df.groupby('cmdb_id').agg(
        count=('timestamp','size'),
        earliest=('timestamp','min'),
        latest=('timestamp','max')
    ).reset_index().sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    error_summary = pd.DataFrame(columns=['cmdb_id','count','earliest','latest'])

# sample rows (up to 20) with selected columns
error_sample_rows = error_df[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# Final compact outputs (limited to top 20 where applicable)
metric_cmdb_counts, metric_kpi_counts_top20, metric_p95_top20, trace_cmdb_counts, trace_name_counts_top20, trace_p95_top20, log_cmdb_counts, log_name_counts_top20, log_p95_top20, error_summary, error_sample_rows

```
Out[1]:
```
# Compose a plain-English summary based on the previously computed summaries
top_mem = metric_p95_top20[metric_p95_top20['kpi_name']=='mem'].head(3)
top_diskio = metric_p95_top20[metric_p95_top20['kpi_name']=='diskio'].head(3)
top_log_rows = log_p95_top20[log_p95_top20['log_name']=='log.row_count'].head(3)

lines = []
lines.append("Telemetry summary (concise):")
lines.append("")
# Metric summary - memory
if not top_mem.empty:
    mem_lines = []
    for _, r in top_mem.iterrows():
        mem_lines.append(f"{r['cmdb_id']}: {r['p95']:,g}")
    lines.append("Top memory (P95) by component: " + "; ".join(mem_lines) + ".")
else:
    lines.append("No memory (mem) metrics present.")

# Metric summary - diskio (if present)
if not top_diskio.empty:
    diskio_lines = []
    for _, r in top_diskio.iterrows():
        diskio_lines.append(f"{r['cmdb_id']}: {r['p95']:,g}")
    lines.append("Top disk I/O (P95) by component (where present): " + "; ".join(diskio_lines) + ".")
    
# KPI coverage note
lines.append(f"Metric KPIs present (top counts): cpu, socket, mem (each 375 rows), diskio (225 rows), latency-50/90, workload.")
lines.append("")

# Trace summary
if trace_p95_top20.empty and trace_df.shape[0]==0:
    lines.append("Trace data: none available.")
else:
    lines.append("Trace data: present (see trace summaries).")

lines.append("")

# Log summary
if not top_log_rows.empty:
    log_lines = []
    for _, r in top_log_rows.iterrows():
        log_lines.append(f"{r['cmdb_id']}: {r['p95']:,g}")
    lines.append("Top log row-count (P95): " + "; ".join(log_lines) + ".")
else:
    lines.append("No log.row_count entries present.")

# log.error_count observation
if 'log.error_count' in log_name_counts_top20['log_name'].values:
    # check if error counts are non-zero in p95 table
    err_nonzero = (log_p95_top20[(log_p95_top20['log_name']=='log.error_count') & (log_p95_top20['p95']>0)].shape[0] > 0)
    if err_nonzero:
        lines.append("Some log.error_count P95 values > 0 exist.")
    else:
        lines.append("log.error_count P95 values are zero for listed components (no sustained logged errors).")
else:
    lines.append("No log.error_count metric present.")

lines.append("")

# error logs
if error_df.empty:
    lines.append("error_logs.csv: no error log rows found.")
else:
    # summarize top cmdbs
    top_err = error_summary[['cmdb_id','count','earliest','latest']]
    lines.append("error_logs: counts per component present (see summary).")

lines.append("")
lines.append("Overall conclusion: The strongest signal is very high P95 memory values across multiple services (highest: carts, catalogue-db, orders, shipping, queue-master). Additionally, front-end shows very high log throughput (log.row_count P95 ~2094.8), and other services (user, queue-master) also show elevated log row counts. There are no trace records and no recorded error log messages in error_logs.csv to confirm an application error; investigate memory pressure and front-end/log throughput further as likely contributors.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  count
0               carts    175
1              orders    175
2           catalogue    150
3           front-end    150
4                user    150
5            shipping    150
6             payment    150
7           orders-db    100
8        catalogue-db    100
9            carts-db    100
10       queue-master    100
11         session-db    100
12           rabbitmq    100
13            user-db    100
14  rabbitmq-exporter     75,      kpi_name  count
0         cpu    375
1      socket    375
2         mem    375
3      diskio    225
4  latency-50    175
5  latency-90    175
6    workload    175,               cmdb_id kpi_name           p95
0               carts      mem  1.105144e+09
1        catalogue-db      mem  4.310381e+08
2              orders      mem  3.185497e+08
3            shipping      mem  3.049593e+08
4        queue-master      mem  2.706979e+08
5           front-end      mem  1.051499e+08
6            rabbitmq      mem  9.356670e+07
7            carts-db      mem  8.025343e+07
8             user-db      mem  4.502673e+07
9           orders-db      mem  4.063390e+07
10               user      mem  8.015776e+06
11          catalogue      mem  6.108692e+06
12  rabbitmq-exporter      mem  4.616192e+06
13            payment      mem  4.470634e+06
14         session-db      mem  2.917248e+06
15           carts-db   diskio  2.230157e+06
16            user-db   diskio  1.554148e+06
17          orders-db   diskio  9.487070e+05
18       queue-master   diskio  5.881183e+04
19         session-db   diskio  1.770564e+04, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, p95]
Index: [],          cmdb_id  count
0      catalogue     50
1         orders     50
2      front-end     50
3       shipping     50
4           user     50
5        payment     50
6   queue-master     50
7       carts-db     48
8          carts     48
9     session-db     10
10       user-db      4,           log_name  count
0  log.error_count    230
1    log.row_count    230,          cmdb_id         log_name      p95
0      front-end    log.row_count  2094.80
1           user    log.row_count   632.60
2   queue-master    log.row_count   597.60
3      catalogue    log.row_count   170.80
4         orders    log.row_count   132.80
5          carts    log.row_count   121.40
6        payment    log.row_count   106.40
7       shipping    log.row_count    66.40
8     session-db    log.row_count     5.00
9        user-db    log.row_count     1.95
10      carts-db    log.row_count     1.00
11     catalogue  log.error_count     0.00
12         carts  log.error_count     0.00
13      carts-db  log.error_count     0.00
14  queue-master  log.error_count     0.00
15       payment  log.error_count     0.00
16     front-end  log.error_count     0.00
17        orders  log.error_count     0.00
18      shipping  log.error_count     0.00
19    session-db  log.error_count     0.00, Empty DataFrame
Columns: [cmdb_id, count, earliest, latest]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reload CSVs to preserve original integer timestamps in 'timestamp' column
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Preserve original integer timestamp columns and create datetime columns for filtering (UTC)
for df in (metric_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')  # keep original ints (nullable)
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)

# Ensure numeric 'value' columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window (UTC)
start_dt = pd.to_datetime("2024-01-20 02:57:02", utc=True)
end_dt   = pd.to_datetime("2024-01-20 03:27:02", utc=True)

# Target components
targets = ['carts','catalogue','orders','payment','user']

# --- 1) Metrics anomaly scan ---

# Compute global P95 per (cmdb_id, kpi_name) using entire metric_df
metric_global_p95 = (
    metric_df.dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Restrict to target components' pairs
metric_pairs = metric_global_p95[metric_global_p95['cmdb_id'].isin(targets)].copy()

# Filter metric_df to incident window and target components
metric_window = metric_df[
    (metric_df['timestamp_dt'] >= start_dt) &
    (metric_df['timestamp_dt'] <= end_dt) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# Right-join window rows to thresholds so we keep pairs with no rows in window
if not metric_pairs.empty:
    metric_window_join = metric_window.merge(metric_pairs, on=['cmdb_id','kpi_name'], how='right', suffixes=('_row','_th'))
else:
    metric_window_join = pd.DataFrame(columns=list(metric_df.columns) + ['global_p95'])

# Determine breaches: value >= global_p95 where both present
metric_window_join['is_breach'] = False
mask_valid = metric_window_join['value'].notna() & metric_window_join['global_p95'].notna()
metric_window_join.loc[mask_valid, 'is_breach'] = metric_window_join.loc[mask_valid, 'value'] >= metric_window_join.loc[mask_valid, 'global_p95']

# Aggregate breaches per (cmdb_id, kpi_name)
breaches = (
    metric_window_join[metric_window_join['is_breach']]
    .groupby(['cmdb_id','kpi_name'])
    .agg(
        count_of_breaches=('is_breach','sum'),
        earliest_breach_timestamp=('timestamp','min')
    )
    .reset_index()
)

# Compute max_value_in_window per pair (if any rows in window)
max_in_window = metric_window.groupby(['cmdb_id','kpi_name'])['value'].max().reset_index(name='max_value_in_window')

# Combine thresholds (metric_pairs), breaches, and max_in_window to produce final table including zero-breach entries
metrics_agg = metric_pairs.merge(breaches, on=['cmdb_id','kpi_name'], how='left')
metrics_agg = metrics_agg.merge(max_in_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for count_of_breaches with 0 and ensure earliest timestamp stays as original integer or NA
metrics_agg['count_of_breaches'] = metrics_agg['count_of_breaches'].fillna(0).astype(int)
metrics_agg['earliest_breach_timestamp'] = metrics_agg['earliest_breach_timestamp'].where(metrics_agg['earliest_breach_timestamp'].notna(), pd.NA)

# Determine rows_in_window and note if no rows
pairs_window_counts = metric_window.groupby(['cmdb_id','kpi_name']).size().reset_index(name='rows_in_window')
metrics_agg = metrics_agg.merge(pairs_window_counts, on=['cmdb_id','kpi_name'], how='left')
metrics_agg['rows_in_window'] = metrics_agg['rows_in_window'].fillna(0).astype(int)
metrics_agg['note'] = pd.NA
metrics_agg.loc[metrics_agg['rows_in_window']==0, 'note'] = 'no rows in window'

# Final metrics anomalies table with requested columns, limited to top 20 sorted by count desc
metrics_anomalies = metrics_agg[['cmdb_id','kpi_name','count_of_breaches','earliest_breach_timestamp','max_value_in_window','global_p95','rows_in_window','note']]
metrics_anomalies = metrics_anomalies.sort_values(['count_of_breaches','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)


# --- 2) Logs anomaly scan ---

# Compute global P95 per (cmdb_id, log_name) using full log_df
log_global_p95 = (
    log_df.dropna(subset=['value'])
    .groupby(['cmdb_id','log_name'])['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# We check only log.row_count and log.error_count for target components
log_keys = ['log.row_count','log.error_count']
pairs_list = [(c, ln) for c in targets for ln in log_keys]
pairs_df = pd.DataFrame(pairs_list, columns=['cmdb_id','log_name'])
pairs_df = pairs_df.merge(log_global_p95, on=['cmdb_id','log_name'], how='left')

# Filter log_df to window and targets
log_window = log_df[
    (log_df['timestamp_dt'] >= start_dt) &
    (log_df['timestamp_dt'] <= end_dt) &
    (log_df['cmdb_id'].isin(targets))
].copy()

# Right-join to keep all requested pairs even if no rows in window
log_window_join = log_window.merge(pairs_df, on=['cmdb_id','log_name'], how='right', suffixes=('_row','_th'))

# Determine breaches where value >= global_p95 (only when both present)
log_window_join['is_breach'] = False
mask_valid_log = log_window_join['value'].notna() & log_window_join['global_p95'].notna()
log_window_join.loc[mask_valid_log, 'is_breach'] = log_window_join.loc[mask_valid_log, 'value'] >= log_window_join.loc[mask_valid_log, 'global_p95']

# Aggregate breaches per pair
log_breaches = (
    log_window_join[log_window_join['is_breach']]
    .groupby(['cmdb_id','log_name'])
    .agg(
        count_of_breaches=('is_breach','sum'),
        earliest_breach_timestamp=('timestamp','min')
    )
    .reset_index()
)

# Compute max_value_in_window per pair (if any rows in window)
log_max_in_window = log_window.groupby(['cmdb_id','log_name'])['value'].max().reset_index(name='max_value_in_window')

# Combine pairs_df with breach and max info
logs_agg = pairs_df.merge(log_breaches, on=['cmdb_id','log_name'], how='left')
logs_agg = logs_agg.merge(log_max_in_window, on=['cmdb_id','log_name'], how='left')

logs_agg['count_of_breaches'] = logs_agg['count_of_breaches'].fillna(0).astype(int)
logs_agg['earliest_breach_timestamp'] = logs_agg['earliest_breach_timestamp'].where(logs_agg['earliest_breach_timestamp'].notna(), pd.NA)

# Note which pairs had no global threshold (i.e., missing globally)
logs_agg['note'] = pd.NA
logs_agg.loc[logs_agg['global_p95'].isna(), 'note'] = 'no global metric for this pair'

# Mark rows_in_window and note if no rows
rows_in_window_logs = log_window.groupby(['cmdb_id','log_name']).size().reset_index(name='rows_in_window')
logs_agg = logs_agg.merge(rows_in_window_logs, on=['cmdb_id','log_name'], how='left')
logs_agg['rows_in_window'] = logs_agg['rows_in_window'].fillna(0).astype(int)

mask_no_rows = logs_agg['rows_in_window']==0
# For pairs with no rows, set note to existing note or 'no rows in window' if blank
logs_agg.loc[mask_no_rows, 'note'] = logs_agg.loc[mask_no_rows, 'note'].fillna('no rows in window')

# Final logs anomalies table
logs_anomalies = logs_agg[['cmdb_id','log_name','count_of_breaches','earliest_breach_timestamp','max_value_in_window','global_p95','rows_in_window','note']]
logs_anomalies = logs_anomalies.sort_values(['count_of_breaches','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)


# --- 3) error_logs check ---

# Filter error logs to window and target components
error_window = error_df[
    (error_df['timestamp_dt'] >= start_dt) &
    (error_df['timestamp_dt'] <= end_dt) &
    (error_df['cmdb_id'].isin(targets))
].copy()

# Counts per cmdb_id (top 20) and sample rows (original timestamp, cmdb_id, message)
if not error_window.empty:
    error_counts = error_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
    error_samples = error_window[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)
else:
    error_counts = pd.DataFrame(columns=['cmdb_id','count'])
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Return compact results (top 20 where applicable)
metrics_anomalies, logs_anomalies, error_counts, error_samples

```
Out[2]:
```
summary = (
    "Incident window: 2024-01-20 02:57:02 → 2024-01-20 03:27:02 UTC\n\n"
    "Key findings (compact):\n"
    "- Metrics anomalies:\n"
    "  * catalogue.socket: 25 breaches (global_p95=7.0, max_in_window=7.0, earliest_breach_ts=1705719600)\n"
    "  * carts.diskio and orders.diskio: 25 breaches each (global_p95=0, max_in_window=0) — repeated at every row in window\n"
    "  * payment.socket: 24 breaches (global_p95=6.0, max_in_window=6.0)\n"
    "  * user.socket: 5 breaches (global_p95=19.0, max_in_window=19.0)\n"
    "  * Memory (mem) exceedances exist but are infrequent (2 breaches each for carts, orders, user, catalogue, payment; p95 thresholds slightly below observed maxima).\n"
    "  * Several cpu/workload pairs show 2 breaches each (smaller, short spikes).\n\n"
    "- Logs anomalies:\n"
    "  * log.error_count shows 'breaches' equal to global_p95 (global_p95=0 and max_in_window=0) for multiple components — i.e., no non-zero error counts observed.\n"
    "  * log.row_count exceeded global P95 briefly for several services (each with 2 breaches):\n"
    "    - user: max 654 > p95 632.6\n"
    "    - catalogue: max 172 > p95 170.8\n"
    "    - orders: max 138 > p95 132.8\n"
    "    - carts: max 125 > p95 121.4\n"
    "    - payment: max 109 > p95 106.4\n\n"
    "- error_logs.csv: no error_log rows found for the target components in the incident window (no sample rows).\n\n"
    "Interpretation & next steps (practical):\n"
    "- The dominant signals are frequent socket threshold hits (catalogue, payment, user) and repeated diskio-==0 readings for carts/orders. Investigate network/socket conditions and verify disk I/O metric semantics (the p95==0 and max==0 could indicate metric artifacts or no activity).\n"
    "- Memory spikes occurred (notably carts and orders) but were rare during the window — worth checking historical context and heap usage around those timestamps.\n"
    "- Elevated log throughput (log.row_count) in user and other services suggests increased request/processing volume; this may relate to resource pressure.\n"
    "- No application error messages were recorded in error_logs for these services during the window.\n\n"
    "Summary conclusion: focus initial investigation on socket behavior (catalogue/payment/user), verify disk I/O metric validity for carts/orders, and review memory and log throughput spikes (especially for carts and user). No explicit error-log evidence of application failures in the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  kpi_name  count_of_breaches  earliest_breach_timestamp  max_value_in_window    global_p95  rows_in_window  note
0   catalogue    socket                 25                 1705719600         7.000000e+00  7.000000e+00              25  <NA>
1       carts    diskio                 25                 1705719600         0.000000e+00  0.000000e+00              25  <NA>
2      orders    diskio                 25                 1705719600         0.000000e+00  0.000000e+00              25  <NA>
3     payment    socket                 24                 1705719660         6.000000e+00  6.000000e+00              25  <NA>
4        user    socket                  5                 1705720800         1.900000e+01  1.900000e+01              25  <NA>
5       carts       mem                  2                 1705720560         1.243691e+09  1.105144e+09              25  <NA>
6      orders       mem                  2                 1705720980         3.186127e+08  3.185497e+08              25  <NA>
7        user       mem                  2                 1705720980         8.105984e+06  8.015776e+06              25  <NA>
8   catalogue       mem                  2                 1705720980         6.123520e+06  6.108692e+06              25  <NA>
9     payment       mem                  2                 1705720980         4.485120e+06  4.470634e+06              25  <NA>
10      carts       cpu                  2                 1705720980         7.999322e+01  7.911495e+01              25  <NA>
11       user  workload                  2                 1705720380         2.086700e+01  1.995100e+01              25  <NA>
12     orders    socket                  2                 1705720320         1.753333e+01  1.739000e+01              25  <NA>
13      carts    socket                  2                 1705720800         1.600000e+01  1.536000e+01              25  <NA>
14      carts  workload                  2                 1705720380         8.703300e+00  8.542933e+00              25  <NA>
15  catalogue  workload                  2                 1705720980         4.533000e+00  4.364930e+00              25  <NA>
16     orders       cpu                  2                 1705720620         3.236221e+00  2.059798e+00              25  <NA>
17     orders  workload                  2                 1705720380         2.333000e+00  2.216517e+00              25  <NA>
18    payment  workload                  2                 1705720380         2.333000e+00  2.237120e+00              25  <NA>
19       user       cpu                  2                 1705720380         1.186790e+00  1.099415e+00              25  <NA>,      cmdb_id         log_name  count_of_breaches  earliest_breach_timestamp  max_value_in_window  global_p95  rows_in_window  note
0  catalogue  log.error_count                 25                 1705719600                    0         0.0              25  <NA>
1     orders  log.error_count                 25                 1705719600                    0         0.0              25  <NA>
2    payment  log.error_count                 25                 1705719600                    0         0.0              25  <NA>
3       user  log.error_count                 25                 1705719600                    0         0.0              25  <NA>
4      carts  log.error_count                 24                 1705719600                    0         0.0              24  <NA>
5       user    log.row_count                  2                 1705720320                  654       632.6              25  <NA>
6  catalogue    log.row_count                  2                 1705720020                  172       170.8              25  <NA>
7     orders    log.row_count                  2                 1705720320                  138       132.8              25  <NA>
8      carts    log.row_count                  2                 1705719780                  125       121.4              24  <NA>
9    payment    log.row_count                  2                 1705720320                  109       106.4              25  <NA>, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing variables from prior runs: metric_df, log_df, metric_global_p95, log_global_p95, start_dt, end_dt, targets
# If any are missing (e.g., running this cell standalone), reload minimal sources:
try:
    _ = metric_df.shape
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    _ = log_df.shape
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
    log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'].astype('int64'), unit='s', utc=True)
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

try:
    _ = metric_global_p95.shape
except NameError:
    metric_global_p95 = (
        metric_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(0.95)
        .reset_index(name='global_p95')
    )

try:
    _ = log_global_p95.shape
except NameError:
    log_global_p95 = (
        log_df.dropna(subset=['value'])
        .groupby(['cmdb_id', 'log_name'])['value']
        .quantile(0.95)
        .reset_index(name='global_p95')
    )

# Incident window (ensure existing)
try:
    _ = start_dt, end_dt
except NameError:
    start_dt = pd.to_datetime("2024-01-20 02:57:02", utc=True)
    end_dt   = pd.to_datetime("2024-01-20 03:27:02", utc=True)

# Target components and KPI/log keys to analyze (from previous scan)
metric_kpis = ['socket','diskio','mem','cpu','workload']
log_keys = ['log.row_count','log.error_count']
targets = ['carts','catalogue','orders','payment','user']

# Helper to find consecutive fault intervals for a series (rows sorted by timestamp)
def find_fault_intervals(series_df, global_p95_value):
    """
    series_df: DataFrame with columns ['timestamp','timestamp_dt','value'] sorted by timestamp ascending
    global_p95_value: numeric threshold (may be NaN)
    Returns list of dicts representing fault intervals where value >= global_p95_value
    """
    out = []
    if pd.isna(global_p95_value):
        return out  # cannot evaluate breaches without a global threshold

    if series_df.empty:
        return out

    # Determine expected step (minimum positive diff) in seconds for this series
    ts = series_df['timestamp'].astype('int64')
    diffs = ts.diff().dropna()
    pos_diffs = diffs[diffs > 0]
    expected_step = int(pos_diffs.min()) if not pos_diffs.empty else None

    # Identify breach rows
    breaches = series_df[series_df['value'] >= global_p95_value].copy()
    if breaches.empty:
        return out

    # If only one breach row, it's a single interval
    if expected_step is None:
        # treat all breaches as isolated rows (no concept of adjacency)
        for _, row in breaches.iterrows():
            out.append((row['timestamp'], row['timestamp'], 1, series_df.shape[0], row['value']))
        return out

    # For breaches, compute diff and split into consecutive groups where diff == expected_step
    breaches = breaches.sort_values('timestamp').reset_index(drop=True)
    breaches['ts_diff'] = breaches['timestamp'].astype('int64').diff().fillna(expected_step).astype('Int64')
    # Start a new group when ts_diff != expected_step
    breaches['group'] = (breaches['ts_diff'] != expected_step).cumsum()

    grouped = breaches.groupby('group')
    for _, g in grouped:
        start_ts = int(g['timestamp'].iloc[0])
        end_ts = int(g['timestamp'].iloc[-1])
        duration_in_rows = g.shape[0]
        max_value = float(g['value'].max())
        out.append((start_ts, end_ts, duration_in_rows, series_df.shape[0], max_value))
    return out

# Collect faults from metric KPIs
faults = []

for cmdb in targets:
    for kpi in metric_kpis:
        # global threshold from precomputed metric_global_p95
        row_th = metric_global_p95[
            (metric_global_p95['cmdb_id']==cmdb) & (metric_global_p95['kpi_name']==kpi)
        ]
        global_p95_value = float(row_th['global_p95'].iloc[0]) if not row_th.empty else np.nan

        # filter series to window
        series = metric_df[
            (metric_df['cmdb_id']==cmdb) &
            (metric_df['kpi_name']==kpi) &
            (metric_df['timestamp_dt'] >= start_dt) &
            (metric_df['timestamp_dt'] <= end_dt)
        ].sort_values('timestamp')

        # find fault intervals where value >= global_p95_value
        intervals = find_fault_intervals(series, global_p95_value)

        # For each interval, produce required row
        for (start_ts, end_ts, duration_rows, rows_in_window, max_val) in intervals:
            if pd.isna(global_p95_value):
                severity = np.nan
                note = 'no global_p95'
            elif global_p95_value == 0:
                severity = np.inf  # flag as infinite severity for sorting; note will indicate p95==0
                note = 'global_p95==0 (possible metric artifact)'
            else:
                severity = max_val / global_p95_value
                note = pd.NA

            faults.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'fault_start_timestamp': int(start_ts),
                'fault_end_timestamp': int(end_ts),
                'duration_in_rows': int(duration_rows),
                'rows_in_window_for_series': int(rows_in_window),
                'max_value_in_fault': float(max_val),
                'global_p95': (np.nan if pd.isna(global_p95_value) else float(global_p95_value)),
                'severity_ratio': severity,
                'earliest_breach_timestamp': int(start_ts),
                'note': note
            })

# Collect faults from log keys
for cmdb in targets:
    for lk in log_keys:
        row_th = log_global_p95[
            (log_global_p95['cmdb_id']==cmdb) & (log_global_p95['log_name']==lk)
        ]
        global_p95_value = float(row_th['global_p95'].iloc[0]) if not row_th.empty else np.nan

        series = log_df[
            (log_df['cmdb_id']==cmdb) &
            (log_df['log_name']==lk) &
            (log_df['timestamp_dt'] >= start_dt) &
            (log_df['timestamp_dt'] <= end_dt)
        ].sort_values('timestamp')

        intervals = find_fault_intervals(series, global_p95_value)

        for (start_ts, end_ts, duration_rows, rows_in_window, max_val) in intervals:
            if pd.isna(global_p95_value):
                severity = np.nan
                note = 'no global_p95'
            elif global_p95_value == 0:
                severity = np.inf
                note = 'global_p95==0 (possible metric artifact)'
            else:
                severity = max_val / global_p95_value
                note = pd.NA

            faults.append({
                'cmdb_id': cmdb,
                'kpi_name': lk,
                'fault_start_timestamp': int(start_ts),
                'fault_end_timestamp': int(end_ts),
                'duration_in_rows': int(duration_rows),
                'rows_in_window_for_series': int(rows_in_window),
                'max_value_in_fault': float(max_val),
                'global_p95': (np.nan if pd.isna(global_p95_value) else float(global_p95_value)),
                'severity_ratio': severity,
                'earliest_breach_timestamp': int(start_ts),
                'note': note
            })

# Build DataFrame of faults
if faults:
    faults_df = pd.DataFrame(faults)
else:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp','duration_in_rows',
        'rows_in_window_for_series','max_value_in_fault','global_p95','severity_ratio',
        'earliest_breach_timestamp','note'
    ])

# Normalize data types and sort: severity_ratio desc (inf top), then duration desc
# Ensure severity_ratio numeric (np.inf allowed) for sorting
faults_df['severity_ratio_sort'] = faults_df['severity_ratio'].replace({np.nan: -np.inf})
# Put actual np.inf for p95==0; keep note already set
faults_df = faults_df.sort_values(by=['severity_ratio_sort','duration_in_rows'], ascending=[False, False]).head(20)

# Clean up display columns: keep requested columns and present severity_ratio as-is (np.inf indicates p95==0)
display_cols = [
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp','duration_in_rows',
    'rows_in_window_for_series','max_value_in_fault','global_p95','severity_ratio',
    'earliest_breach_timestamp','note'
]
faults_df = faults_df[display_cols].reset_index(drop=True)

# Final compact table (top 20 faults)
faults_df

```
Out[3]:
```
Incident window: 2024-01-20 02:57:02 → 2024-01-20 03:27:02 UTC

Top fault intervals (compact summary of top 20 faults, sorted by severity_ratio then duration):

- Highest-severity (global_p95 == 0 — flagged as possible metric artifact)
  1. carts, diskio — fault_start 1705719600 to fault_end 1705721040, duration 25 rows (rows_in_window 25), max_value_in_fault 0.0, global_p95 0.0, severity_ratio = inf. Note: global_p95==0 (possible metric artifact).
  2. orders, diskio — same full-window interval (1705719600 → 1705721040), duration 25, max=0.0, global_p95=0.0, severity_ratio = inf. Note: global_p95==0 (possible metric artifact).
  3–6. catalogue/orders/payment/user, log.error_count — full-window intervals with max=0.0 and global_p95=0.0 (duration 25 rows for most; carts had 24 rows) → severity_ratio = inf and note global_p95==0 (possible metric artifact).

- Highest true severity ratios (global_p95 > 0)
  7. orders, cpu — single-row fault at 1705720740, duration 1 (rows_in_window 25), max 3.236221, global_p95 2.059798, severity_ratio ≈ 1.571.
  8. carts, mem — single-row fault at 1705720560, duration 1 (rows_in_window 25), max 1.243691e9, global_p95 1.105144e9, severity_ratio ≈ 1.125.
  9. user, cpu — single-row fault at 1705721040, duration 1, max 1.18679, global_p95 1.099415, severity_ratio ≈ 1.079.
  10. orders, workload — single-row fault at 1705721040, severity_ratio ≈ 1.053.
  11. user, workload — single-row fault at 1705721040, severity_ratio ≈ 1.046.
  12. payment, workload — single-row at 1705721040, severity_ratio ≈ 1.043.
  13. carts, socket — small 2-row interval (1705720800 → 1705720860), max 16.0, global_p95 15.36, severity_ratio ≈ 1.042.
  14–19. Several log.row_count faults (orders, catalogue, user, carts) each single-row, severity_ratios ≈ 1.03–1.04 (e.g., user max 654 vs p95 632.6 → ≈1.034).

Notes and interpretation (actionable):
- Several series have global_p95 == 0 (diskio for carts/orders and log.error_count for many services). Those produce infinite severity_ratio and are flagged as possible metric artifacts or indicating no baseline activity — investigate metric collection/semantics for diskio and error_count (0 p95 suggests either no activity historically or incorrect normalization). Do not assume true resource saturation from those zero-p95 hits without confirming metric meaning.
- Non-artifact breaches (severity_ratio > 1) are small spikes (mostly single-row) for cpu, mem, workload, socket and log.row_count. The largest true ratio is orders.cpu (~1.57), then carts.mem (~1.13). Many other breaches are marginal (≈1.03–1.08).
- Some breaches span multiple consecutive rows only for diskio and log.error_count (the full or near-full window) — again likely artifact if values are 0 and threshold is 0.
- log.row_count breaches indicate brief increases in log throughput (user, catalogue, orders, carts, payment) which may reflect short traffic spikes and correspond to some KPI spikes (cpu, workload).

Recommended next steps:
1. Immediately verify diskio and log.error_count metric definitions and collection (global_p95==0 flagged). Confirm whether 0 means no activity or indicates a telemetry/normalization bug.
2. Investigate the short but real spikes:
   - orders.cpu (single spike, severity ~1.57) and carts.mem (single spike) — examine traces, request rates, and GC/heap metrics around the listed timestamps.
   - socket hits for catalogue/payment/user (catalogue had many socket breaches in earlier scan) — check networking/connection logs and front-end load patterns.
3. Correlate brief log.row_count increases (user and others) with request/traffic surge to see if transient load caused the resource spikes.
4. No error_logs sample rows were found in the window, so there is no direct application-error evidence in error_logs — focus on metrics and logs/throughput correlation.

Compact conclusion: confirm and fix potential metric artifacts for diskio and log.error_count; then focus troubleshooting on brief CPU/memory spikes (orders, carts) and socket-related anomalies (catalogue/payment/user) correlated with short log throughput increases.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         kpi_name  fault_start_timestamp  fault_end_timestamp  duration_in_rows  rows_in_window_for_series  max_value_in_fault    global_p95  severity_ratio  earliest_breach_timestamp                                      note
0       carts           diskio             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
1      orders           diskio             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
2   catalogue  log.error_count             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
3      orders  log.error_count             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
4     payment  log.error_count             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
5        user  log.error_count             1705719600           1705721040                25                         25        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
6       carts  log.error_count             1705719600           1705720980                24                         24        0.000000e+00  0.000000e+00             inf                 1705719600  global_p95==0 (possible metric artifact)
7      orders              cpu             1705720740           1705720740                 1                         25        3.236221e+00  2.059798e+00        1.571136                 1705720740                                      <NA>
8       carts              mem             1705720560           1705720560                 1                         25        1.243691e+09  1.105144e+09        1.125366                 1705720560                                      <NA>
9        user              cpu             1705721040           1705721040                 1                         25        1.186790e+00  1.099415e+00        1.079474                 1705721040                                      <NA>
10     orders         workload             1705721040           1705721040                 1                         25        2.333000e+00  2.216517e+00        1.052552                 1705721040                                      <NA>
11       user         workload             1705721040           1705721040                 1                         25        2.086700e+01  1.995100e+01        1.045912                 1705721040                                      <NA>
12    payment         workload             1705721040           1705721040                 1                         25        2.333000e+00  2.237120e+00        1.042859                 1705721040                                      <NA>
13      carts           socket             1705720800           1705720860                 2                         25        1.600000e+01  1.536000e+01        1.041667                 1705720800                                      <NA>
14     orders    log.row_count             1705720320           1705720320                 1                         25        1.380000e+02  1.328000e+02        1.039157                 1705720320                                      <NA>
15  catalogue         workload             1705720980           1705721040                 2                         25        4.533000e+00  4.364930e+00        1.038505                 1705720980                                      <NA>
16       user    log.row_count             1705720320           1705720320                 1                         25        6.540000e+02  6.326000e+02        1.033829                 1705720320                                      <NA>
17     orders              cpu             1705720620           1705720620                 1                         25        2.121715e+00  2.059798e+00        1.030060                 1705720620                                      <NA>
18      carts    log.row_count             1705719780           1705719780                 1                         24        1.250000e+02  1.214000e+02        1.029654                 1705719780                                      <NA>
19      carts    log.row_count             1705720260           1705720260                 1                         24        1.250000e+02  1.214000e+02        1.029654                 1705720260                                      <NA>```
```